<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/kinematics/01_forward_kinematics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K1 2リンク平面アームの順運動学(notebook)

本文:[`01-forward-kinematics.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/kinematics/01-forward-kinematics.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

l1, l2 = 1.0, 0.8   # リンクの長さ(この単元ではずっとこの値を使う)

def draw_arm(ax, elbow, hand, **kw):
    # ベース(原点)・肘・手先を線で結んで、アームを描く
    pts = np.array([[0.0, 0.0], elbow, hand])
    ax.plot(pts[:, 0], pts[:, 1], "-o", **kw)

## 1. 肘と手先の位置

**予想**:$q = (\pi/2, -\pi/2)$ のとき、肘と手先はどこにあるか。アームの図を紙に描いて予想する。

- 予想:(ここに書く)

**手計算**:本文の確認課題1 の4つの関節角 $(0, 0)$、$(\pi/2, -\pi/2)$、$(0, \pi/2)$、$(\pi/2, \pi/2)$ について、肘と手先の位置を求める。

- 手計算:(ここに書く)

**実装**:本文の式 (1) で、肘の位置 `elbow` と手先の位置 `hand` を返す関数 `fk_points` を書く。

In [ ]:
def fk_points(q, l1=l1, l2=l2):
    q1, q2 = q
    raise NotImplementedError("TODO: 肘の位置 elbow と手先の位置 hand(どちらも長さ 2 の配列)を計算する")
    return elbow, hand

check("q=(0, 0) の手先", fk_points([0.0, 0.0])[1], [1.8, 0.0])
check("q=(π/2, -π/2) の肘", fk_points([np.pi / 2, -np.pi / 2])[0], [0.0, 1.0])
check("q=(π/2, -π/2) の手先", fk_points([np.pi / 2, -np.pi / 2])[1], [0.8, 1.0])
check("q=(0, π/2) の手先(M3 §4 と同じ)", fk_points([0.0, np.pi / 2])[1], [1.0, 0.8])
check("q=(π/2, π/2) の手先", fk_points([np.pi / 2, np.pi / 2])[1], [-0.8, 1.0])

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for q in [(0.0, 0.0), (np.pi / 2, -np.pi / 2), (0.0, np.pi / 2), (np.pi / 2, np.pi / 2)]:
    elbow, hand = fk_points(q)
    draw_arm(ax, elbow, hand, label=f"q=({q[0]:.2f}, {q[1]:.2f})")
ax.set_aspect("equal"); ax.grid(True); ax.legend(fontsize=8)
ax.set_xlabel("x"); ax.set_ylabel("y"); plt.show()

図が手計算と合っているかを確かめる。$q = (\pi/2, \pi/2)$ では、第2リンクの向きは x 軸から $q_1 + q_2 = \pi$(左向き)になる。$q_2$ が相対角なので、角度が足し算になる。

## 2. 回転行列で書く

**予想**:$q_2 = \pi/2$ に固定して、$q_1$ を $0$ から $2\pi$ まで回すと、手先はどんな線を描くか。

- 予想:(ここに書く)

**実装**:本文の式 (2) で手先の位置を返す関数 `fk_rot` を書く。回転行列 `rot` は M1 と同じものを用意してある。100 個のランダムな関節角で、式 (1) の `fk_points` と一致することを確かめる。

In [ ]:
def rot(theta):
    # M1 の回転行列:ベクトルを theta だけ反時計回りに回す
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

def fk_rot(q, l1=l1, l2=l2):
    q1, q2 = q
    raise NotImplementedError("TODO: R(q1) と、ベクトル (l1 + l2 cos q2, l2 sin q2) の積")

rng = np.random.default_rng(0)
qs = rng.uniform(-np.pi, np.pi, size=(100, 2))
check("式 (1) と式 (2) が一致(100 個の関節角)", [fk_rot(q) for q in qs], [fk_points(q)[1] for q in qs])

In [ ]:
q1s = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(5, 5))
for q2 in [0.0, np.pi / 2, 2 * np.pi / 3]:
    pts = np.array([fk_points((q1, q2))[1] for q1 in q1s])
    plt.plot(pts[:, 0], pts[:, 1], label=f"q2 = {q2:.2f} (fixed)")
plt.gca().set_aspect("equal"); plt.grid(True); plt.legend(fontsize=8)
plt.xlabel("x"); plt.ylabel("y"); plt.show()

$q_2$ を固定して $q_1$ だけを回すと、手先は原点を中心とする円を描く。肘を大きく曲げる($q_2$ を大きくする)ほど、円は小さくなる。式 (2) で、$q_1$ は「アーム全体を回す」だけだからである。

## 3. 手先までの距離

**予想**:原点から手先までの距離は、$q_1$ を変えると変わるか。

- 予想:(ここに書く)

**手計算**:$q_2 = 2\pi/3$ のとき、本文の式 (3) で距離 $r$ を求める。

- 手計算:(ここに書く)

**実装**:式 (3) で距離 $r$ を返す関数 `reach` を書く($r^2$ ではなく $r$ を返す)。

In [ ]:
def reach(q2, l1=l1, l2=l2):
    raise NotImplementedError("TODO: 式 (3) の r")

check("q2 = 0 のとき r = l1 + l2", reach(0.0), 1.8)
check("q2 = π のとき r = |l1 - l2|", reach(np.pi), 0.2)
check("100 個の関節角で、手先の位置のノルムと一致", [reach(q[1]) for q in qs], [np.linalg.norm(fk_points(q)[1]) for q in qs])
print(f"q2 = 2π/3 のとき r = {reach(2 * np.pi / 3):.4f}")

## 4. 到達可能領域

**予想**:関節角をランダムに選んで手先の位置に点を打つ。点はどんな形に集まるか。実行する前に、紙に形を描く。

- 予想:(ここに書く)

**実装**:目標の位置 `p` が本文の式 (4) を満たすとき `True` を返す関数 `in_workspace` を書く。

In [ ]:
def in_workspace(p, l1=l1, l2=l2):
    raise NotImplementedError("TODO: |l1 - l2| <= ||p|| <= l1 + l2 なら True")

check("(1.0, 0.8) は届く", in_workspace([1.0, 0.8]), True)
check("(2.5, 0.0) は届かない(外側)", in_workspace([2.5, 0.0]), False)
check("(0.1, 0.0) は届かない(内側の穴)", in_workspace([0.1, 0.0]), False)

In [ ]:
def plot_workspace(q_samples, l1=l1, l2=l2, title=""):
    # 関節角の組ごとに手先の位置を計算して点を打つ。赤の円は外側の限界、緑の円は内側の限界
    pts = np.array([fk_points(q, l1, l2)[1] for q in q_samples])
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.3)
    th = np.linspace(0, 2 * np.pi, 200)
    for r, color in [(l1 + l2, "r"), (abs(l1 - l2), "g")]:
        ax.plot(r * np.cos(th), r * np.sin(th), color, lw=1)
    ax.set_aspect("equal"); ax.grid(True); ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y"); plt.show()
    return pts

rng = np.random.default_rng(1)
q_free = rng.uniform(-np.pi, np.pi, size=(3000, 2))
pts = plot_workspace(q_free, title="both joints rotate freely")
check("すべての点が円環の中にある", all(in_workspace(p) for p in pts), True)

点は、内側の半径 $|l_1 - l_2| = 0.2$、外側の半径 $l_1 + l_2 = 1.8$ の円環に集まる。原点の近くと、半径 1.8 より外には点がない。

## 5. 関節の可動範囲

**予想**:$q_2$ が $0 \le q_2 \le 2\pi/3$ しか動けないとき、到達可能領域はどう変わるか(本文の確認課題5)。

- 予想:(ここに書く)

In [ ]:
rng = np.random.default_rng(2)
q_lim = np.column_stack([rng.uniform(-np.pi, np.pi, 3000), rng.uniform(0.0, 2 * np.pi / 3, 3000)])
pts = plot_workspace(q_lim, title="0 <= q2 <= 2pi/3")
r_min = np.linalg.norm(pts, axis=1).min()
print(f"手先までの距離の最小値:{r_min:.3f}   式 (3) の reach(2π/3) = {reach(2 * np.pi / 3):.3f}")

肘を $2\pi/3$ より深く曲げられないので、手先は原点に近づけない。内側の穴が、半径 0.2 から $r(2\pi/3) \approx 0.917$ まで広がる。

## 6. 条件を変える

1. リンクの長さを $l_1 = l_2 = 1.0$ にすると、内側の穴はどうなるか。手先を原点に置けるのは、どんな関節角のときか(例:`plot_workspace(q_free, l1=1.0, l2=1.0)`)
2. 第2リンクのほうが長い場合($l_1 = 0.5$、$l_2 = 1.3$)、内側の穴の半径はいくつになるか。予想してから確かめる
3. $q_2$ は自由に回し、$q_1$ だけを $0 \le q_1 \le \pi$ に制限すると、手先は x 軸より下に行けるか。予想してから確かめる

In [ ]:
# 1〜3 をここで試す

## 7. 説明(自分の言葉で書く)

1. $q_2$ が相対角であることは、式 (1) と図のどこに表れているか
   - (ここに書く)
2. 原点から手先までの距離が $q_1$ によらないのはなぜか
   - (ここに書く)
3. 強化学習でアームに目標の位置を与えるとき、何を確かめるか。理由とともに書く
   - (ここに書く)